<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile
import numpy as np

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [3]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:00<00:00, 277MB/s] 


In [4]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [5]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [6]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [7]:
len(tickers)

503

In [8]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [9]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_436/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-29  119.264961  181.663010  133.065170  132.250000  97.580269   
2023-08-30  119.910057  185.145889  133.723373  130.610001  98.087563   
2023-08-31  118.336418  185.362946  132.497192  131.550003  96.669006   
2023-09-01  119.157471  186.931717  133.615158  132.690002  96.622025   
2023-09-05  116.049248  187.168533  131.721848  142.289993  94.771309   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-29  72.030563  306.244781  540.570007  173.920105  73.820602  ...   
2023-08-30  73.304764  306.055450  545.359985  172.976990  73.042679  ...   
2023-08-31  73.086060  306.547791  559.340027  173.177032  71.731079  ...   
2023-09-01  73.228699  310.306610  563.210022  174.622391  72.174301  ...   
2023-09-05  72.135162  308.810638  564.880005  173.770508  71.351158  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2023-08-29  3354700  1082900  4614900   9665000  1117300  7329800  1462700   
2023-08-30  2122000  1263500  2583500  11922100  1001300  6813200  1617600   
2023-08-31  4479800  2924300  3983700  17479800  2080900  7392900  1465300   
2023-09-01  3577000  1946900  2848100  14843500  1200800  5752300  1094000   
2023-09-05  4133700  3084700  4170400  18304000  2679700  5987000  1435900   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-29  2213800  280800  2423800  
2023-08-30  2018000  391000  1673500  
2023-08-31  2746000  372800  1893000  
2023-09-01  1463300  311900  1174300  
2023-09-05  1645000  417300  1673000  

[5 rows x 2515 columns]

In [10]:
df.head()

Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-29  119.264961  181.663010  133.065170  132.250000  97.580269   
2023-08-30  119.910057  185.145889  133.723373  130.610001  98.087563   
2023-08-31  118.336418  185.362946  132.497192  131.550003  96.669006   
2023-09-01  119.157471  186.931717  133.615158  132.690002  96.622025   
2023-09-05  116.049248  187.168533  131.721848  142.289993  94.771309   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-29  72.030563  306.244781  540.570007  173.920105  73.820602  ...   
2023-08-30  73.304764  306.055450  545.359985  172.976990  73.042679  ...   
2023-08-31  73.086060  306.547791  559.340027  173.177032  71.731079  ...   
2023-09-01  73.228699  310.306610  563.210022  174.622391  72.174301  ...   
2023-09-05  72.135162  308.810638  564.880005  173.770508  71.351158  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2023-08-29  3354700  1082900  4614900   9665000  1117300  7329800  1462700   
2023-08-30  2122000  1263500  2583500  11922100  1001300  6813200  1617600   
2023-08-31  4479800  2924300  3983700  17479800  2080900  7392900  1465300   
2023-09-01  3577000  1946900  2848100  14843500  1200800  5752300  1094000   
2023-09-05  4133700  3084700  4170400  18304000  2679700  5987000  1435900   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-29  2213800  280800  2423800  
2023-08-30  2018000  391000  1673500  
2023-08-31  2746000  372800  1893000  
2023-09-01  1463300  311900  1174300  
2023-09-05  1645000  417300  1673000  

[5 rows x 2515 columns]

In [11]:
closes = df["Close"]
returns = closes.pct_change() *100

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
fdf = pd.concat([cs,rs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_436/2048977385.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return
495,2023-08-30,A,119.910057,0.540893
496,2023-08-30,AAPL,185.145889,1.917220
497,2023-08-30,ABBV,133.723373,0.494647
498,2023-08-30,ABNB,130.610001,-1.240075
499,2023-08-30,ABT,98.087563,0.519873
500,2023-08-30,ACGL,73.304764,1.768972
501,2023-08-30,ACN,306.055450,-0.061823
502,2023-08-30,ADBE,545.359985,0.886098
503,2023-08-30,ADI,172.976990,-0.542269
504,2023-08-30,ADM,73.042679,-1.053803


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [12]:
fdf.sort_values(['ticker', 'date'], inplace=True)
fdf.head(25)

,date,ticker,close,return
495,2023-08-30,A,119.910057,0.540893
990,2023-08-31,A,118.336418,-1.312349
1485,2023-09-01,A,119.157471,0.693829
1980,2023-09-05,A,116.049248,-2.608500
2475,2023-09-06,A,115.277084,-0.665376
2970,2023-09-07,A,113.635025,-1.424446
3465,2023-09-08,A,111.416267,-1.952530
3960,2023-09-11,A,110.546364,-0.780769
4455,2023-09-12,A,110.233597,-0.282928
4950,2023-09-13,A,110.868935,0.576356


In [13]:
df=fdf

In [14]:
len(df)

375506

In [15]:
df['Lag 1'] = df['close'].shift(1)
df['Lag 2'] = df['close'].shift(2)
df['Lag 3'] = df['close'].shift(3)
df['Lag 5'] = df['close'].shift(5)
df.head(200)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-08-30,A,119.910057,0.540893,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312349,119.910057,NaN,NaN,NaN
1485,2023-09-01,A,119.157471,0.693829,118.336418,119.910057,NaN,NaN
1980,2023-09-05,A,116.049248,-2.608500,119.157471,118.336418,119.910057,NaN
2475,2023-09-06,A,115.277084,-0.665376,116.049248,119.157471,118.336418,NaN
...,...,...,...,...,...,...,...,...
97349,2024-06-10,A,130.720917,0.037601,130.671783,130.514587,131.241714,129.119202
97848,2024-06-11,A,129.816879,-0.691578,130.720917,130.671783,130.514587,128.578735
98347,2024-06-12,A,130.848633,0.794776,129.816879,130.720917,130.671783,131.241714
98846,2024-06-13,A,128.303635,-1.944994,130.848633,129.816879,130.720917,130.514587


In [16]:
df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
df.head(25)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-08-30,A,119.910057,0.540893,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312349,119.910057,NaN,NaN,NaN
1485,2023-09-01,A,119.157471,0.693829,118.336418,119.910057,NaN,NaN
1980,2023-09-05,A,116.049248,-2.608500,119.157471,118.336418,119.910057,NaN
2475,2023-09-06,A,115.277084,-0.665376,116.049248,119.157471,118.336418,NaN
2970,2023-09-07,A,113.635025,-1.424446,115.277084,116.049248,119.157471,119.910057
3465,2023-09-08,A,111.416267,-1.952530,113.635025,115.277084,116.049248,118.336418
3960,2023-09-11,A,110.546364,-0.780769,111.416267,113.635025,115.277084,119.157471
4455,2023-09-12,A,110.233597,-0.282928,110.546364,111.416267,113.635025,116.049248
4950,2023-09-13,A,110.868935,0.576356,110.233597,110.546364,111.416267,115.277084


In [17]:
df['SMA 5'] = df['close'].rolling(window=5).mean()
df['SMA 10'] = df['close'].rolling(window=10).mean()
df['SMA 20'] = df['close'].rolling(window=20).mean()
df['SMA 30'] = df['close'].rolling(window=30).mean()
df['SMA 50'] = df['close'].rolling(window=50).mean()

df.head(100)
#df = fdf[fdf['ticker'] == 'AAPL'].copy()
#df.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
495,2023-08-30,A,119.910057,0.540893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312349,119.910057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-01,A,119.157471,0.693829,118.336418,119.910057,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-05,A,116.049248,-2.608500,119.157471,118.336418,119.910057,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-06,A,115.277084,-0.665376,116.049248,119.157471,118.336418,NaN,117.746056,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47591,2024-01-17,A,125.691940,-1.846313,128.056259,128.066086,127.222382,128.203430,127.528464,128.197545,132.247811,130.859849,124.255679
48087,2024-01-18,A,127.987617,1.826431,125.691940,128.056259,128.066086,128.605652,127.404857,128.128870,131.912401,130.918634,124.679965
48583,2024-01-19,A,128.733200,0.582543,127.987617,125.691940,128.056259,127.222382,127.707021,128.150455,131.503601,131.034919,125.148348
49079,2024-01-22,A,130.038010,1.013577,128.733200,127.987617,125.691940,128.066086,128.101405,128.345687,131.238881,131.161071,125.583868


In [18]:
df['volatility 5'] = df['return'].rolling(5).std()
df['volatility 10'] = df['return'].rolling(10).std()
df['volatility 20'] = df['return'].rolling(20).std()
df['volatility 30'] = df['return'].rolling(30).std()
df.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
495,2023-08-30,A,119.910057,0.540893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336418,-1.312349,119.910057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-01,A,119.157471,0.693829,118.336418,119.910057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-05,A,116.049248,-2.608500,119.157471,118.336418,119.910057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-06,A,115.277084,-0.665376,116.049248,119.157471,118.336418,NaN,117.746056,NaN,NaN,NaN,NaN,1.369006,NaN,NaN,NaN


In [19]:
df = df.dropna()
#df = df.drop(columns = 'SMA 100')
df.head()


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
24775,2023-11-08,A,107.135719,-1.040330,108.262001,105.314056,106.773331,100.740280,105.960440,103.367986,105.015815,106.415845,108.702597,2.461420,2.085480,2.164409,1.845693
25271,2023-11-09,A,105.519722,-1.508365,107.135719,108.262001,105.314056,102.317093,106.600966,103.703919,104.957051,106.284129,108.414790,2.735473,2.182335,2.005430,1.846616
25767,2023-11-10,A,106.234680,0.677559,105.519722,107.135719,108.262001,106.773331,106.493236,104.262176,104.857154,106.182109,108.172756,1.835852,2.086770,1.982965,1.852094
26263,2023-11-13,A,105.245483,-0.931143,106.234680,105.519722,107.135719,105.314056,106.479521,104.878213,104.659317,106.069805,107.894516,1.769113,2.024682,1.977988,1.856072
26759,2023-11-14,A,109.309959,3.861901,105.245483,106.234680,105.519722,108.262001,106.689113,105.685233,104.620631,106.113877,107.759730,2.200754,2.228399,2.166073,1.988011


In [20]:
len(df)

372852

In [21]:
df['future return'] = df.groupby('ticker')['return'].shift(-1)
df = df.dropna()
print(len(df))
df.head(len(df))


372349


/tmp/ipykernel_436/2419459239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,future return
24775,2023-11-08,A,107.135719,-1.040330,108.262001,105.314056,106.773331,100.740280,105.960440,103.367986,105.015815,106.415845,108.702597,2.461420,2.085480,2.164409,1.845693,-1.508365
25271,2023-11-09,A,105.519722,-1.508365,107.135719,108.262001,105.314056,102.317093,106.600966,103.703919,104.957051,106.284129,108.414790,2.735473,2.182335,2.005430,1.846616,0.677559
25767,2023-11-10,A,106.234680,0.677559,105.519722,107.135719,108.262001,106.773331,106.493236,104.262176,104.857154,106.182109,108.172756,1.835852,2.086770,1.982965,1.852094,-0.931143
26263,2023-11-13,A,105.245483,-0.931143,106.234680,105.519722,107.135719,105.314056,106.479521,104.878213,104.659317,106.069805,107.894516,1.769113,2.024682,1.977988,1.856072,3.861901
26759,2023-11-14,A,109.309959,3.861901,105.245483,106.234680,105.519722,108.262001,106.689113,105.685233,104.620631,106.113877,107.759730,2.200754,2.228399,2.166073,1.988011,1.782994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373491,2026-08-21,ZTS,77.730003,3.612379,75.019997,76.680000,73.959999,73.790001,75.272000,74.971000,75.642500,75.445232,75.671989,2.672445,2.616024,2.685540,2.319784,-0.720449
373994,2026-08-24,ZTS,77.169998,-0.720449,77.730003,75.019997,76.680000,72.970001,76.112000,75.205000,75.655000,75.521962,75.598659,2.597205,2.517997,2.654497,2.324259,0.155504
374497,2026-08-25,ZTS,77.290001,0.155504,77.169998,77.730003,75.019997,73.959999,76.778000,75.399000,75.644000,75.646387,75.564076,2.628931,2.515249,2.649318,2.297041,0.297575
375000,2026-08-26,ZTS,77.519997,0.297575,77.290001,77.169998,77.730003,76.680000,76.945999,75.796999,75.623500,75.763252,75.526545,2.126562,2.333353,2.647344,2.295711,-3.224974


In [22]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50']
x = df[features]
y = df['future return']



In [23]:
split_index = int(0.8 * len(df))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [24]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
tree = DecisionTreeRegressor(max_depth = 5, random_state=72)

tree.fit(x_train, y_train)
predictions = tree.predict(x_test)

score = mean_squared_error(y_test, predictions)
score

4.9716844094031165

In [25]:
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor(n_estimators = 500, random_state=72)

forest.fit(x_train, y_train)

KeyboardInterrupt: 

In [ ]:
predictions = forest.predict(x_test)
score = mean_squared_error(y_test, predictions)
score

In [26]:
forest = RandomForestRegressor(n_estimators = 50)

forest.fit(x_train, y_train)
predictions = forest.predict(x_test)
score = mean_squared_error(y_test, predictions)
score


5.172378576993575